# TV5 — Benchmark MS1 trên Colab CPU-only

Chọn Runtime → Change runtime type → CPU (không accelerator). Source ref phải đã push code TV5. Chạy smoke trước, rồi full assignment khi đã kiểm tra. API key dùng Colab Secrets `S2_API_KEY`, không gõ trực tiếp. Notebook này chưa được chạy trong Colab khi bàn giao bản local.

In [1]:
from pathlib import Path
import os, sys, subprocess, json, datetime, shutil
assert 'google.colab' in sys.modules or __import__('importlib').util.find_spec('google.colab'), 'Notebook dành cho Google Colab'
if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True)
    assert 'GPU ' not in gpu.stdout, 'Chọn runtime CPU-only trước khi benchmark'
SOURCE_URL = 'https://github.com/LeoTkiet/arxiv-data-science.git'
SOURCE_REF = 'feature/tv5-ms1-benchmark-report'  # phải push nhánh này trước, hoặc dùng commit SHA đã review
SOURCE = Path('/content/arxiv-data-science')
if not SOURCE.exists(): subprocess.run(['git','clone',SOURCE_URL,str(SOURCE)],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','origin',SOURCE_REF],check=True)
subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',str(SOURCE)+'[reporting]'],check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-e', '/content/arxiv-data-science[reporting]'], returncode=0)

In [3]:
from pathlib import Path
import os, sys, subprocess, json, datetime, shutil
import google.colab
SOURCE = Path('/content/arxiv-data-science')
commit = subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip()
assert commit.startswith('0f8d83b'), 'Use the reviewed fixes before measuring'
probe = subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True) if shutil.which('nvidia-smi') else None
assert probe is None or 'GPU ' not in probe.stdout, 'CPU-only runtime required'
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT = Path('/content/ms1-data') / ('cpu-sample-' + stamp)
RESULTS = Path('/content/ms1-runs') / ('cpu-sample-' + stamp)
command = [sys.executable,'-m','arxiv_pipeline.reporting.milestone1.benchmark','--output',str(OUTPUT),'--results',str(RESULTS),'--','--ids','2508.12872','--timeout','30','--retries','1']
completed = subprocess.run(command,cwd=SOURCE)
print('PIPELINE_EXIT', completed.returncode)
if (RESULTS/'benchmark.json').exists():
    report = json.loads((RESULTS/'benchmark.json').read_text())
    report['environment']['is_colab'] = True
    report['environment']['cpu_only_confirmation'] = 'CPU selected in Colab runtime settings; NVIDIA GPU probe passed'
    report['environment']['gpu_probe_stdout'] = probe.stdout if probe else 'nvidia-smi unavailable in CPU runtime'
    report['label'] = 'colab_cpu_sample'
    (RESULTS/'benchmark.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    print('BENCHMARK', json.dumps({k: report[k] for k in ['wall_seconds','ram','disk','statistics_status']},ensure_ascii=False))
if (RESULTS/'statistics.json').exists():
    stats = json.loads((RESULTS/'statistics.json').read_text())
    stats['label'] = 'colab_cpu_sample'
    (RESULTS/'statistics.json').write_text(json.dumps(stats,ensure_ascii=False,indent=2),encoding='utf-8')
    print('COUNTS', stats['counts'])
    print('RATES', stats['rates'])
subprocess.run([sys.executable,'-m','arxiv_pipeline.reporting.milestone1.audit','--output',str(OUTPUT),'--assignment',str(SOURCE/'configs/ms1_group6_ranges.json'),'--save',str(RESULTS/'assignment-audit.json')],cwd=SOURCE)
(RESULTS/'invocation.json').write_text(json.dumps({'command':command,'source_commit':commit,'scope':'one assigned paper; not full assignment'},indent=2),encoding='utf-8')
archive = shutil.make_archive('/content/TV5_MS1_COLAB_' + stamp,'zip',RESULTS)
print('RESULT_ARCHIVE', archive)
from google.colab import files
files.download(archive)


PIPELINE_EXIT 0
BENCHMARK {"wall_seconds": 21.5110041800001, "ram": {"peak_rss_bytes": 41771008, "time_weighted_mean_rss_bytes": 38603951.87300131, "observed_seconds": 21.507839932000024}, "disk": {"initial_bytes": 0, "final_bytes": 10907480, "peak_observed_bytes": 17676608}, "statistics_status": "current_run"}
COUNTS {'papers': 1, 'discovery_success_papers': 1, 'unknown_version_papers': 0, 'expected_versions': 1, 'references': 97}
RATES {'discovery_success': {'numerator': 1, 'denominator': 1, 'value': 1.0}, 'download_success': {'numerator': 1, 'denominator': 1, 'value': 1.0}, 'extract_success': {'numerator': 1, 'denominator': 1, 'value': 1.0}, 'version_record_coverage': {'numerator': 1, 'denominator': 1, 'value': 1.0}, 'source_complete_papers': {'numerator': 1, 'denominator': 1, 'value': 1.0}, 'validation_pass': {'numerator': 1, 'denominator': 1, 'value': 1.0}, 'reference_query_success': {'numerator': 1, 'denominator': 1, 'value': 1.0}, 'reference_nonempty_success': {'numerator': 1, '

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Chọn phạm vi và cấu hình

Full = năm khoảng, 25.000 ID lookup. API pacing/retry khiến lượt full dài; nếu ngắt, giữ corpus trên persistent storage và dùng resume. Không mount Drive rồi lưu kết quả vào Git. Mỗi results phải mới và nằm ngoài output.

In [ ]:
MODE = 'smoke'  # 'smoke' hoặc 'full'
RUN_BENCHMARK = False  # đổi True khi đã kiểm tra cấu hình
RESUME = False
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DATA_ROOT = Path('/content/ms1-data')  # đổi sang thư mục persistent nếu cần resume qua session
OUTPUT = DATA_ROOT / MODE
RESULTS = Path('/content/ms1-runs') / f'{MODE}-{STAMP}'
assignment = json.loads((SOURCE/'configs/ms1_group6_ranges.json').read_text())
args = ['--ids','2508.12872'] if MODE == 'smoke' else [arg for r in assignment['ranges'] for arg in ['--range',r['range']]]
assert MODE in {'smoke','full'}
if RESUME: args += ['--resume']
print({'mode': MODE, 'assigned_inputs': 1 if MODE=='smoke' else assignment['assigned_input_count'], 'output':str(OUTPUT), 'results':str(RESULTS), 'free_disk_bytes':shutil.disk_usage('/content').free})
try:
    from google.colab import userdata
    os.environ['S2_API_KEY'] = userdata.get('S2_API_KEY')
except Exception:
    print('S2_API_KEY không có; dùng pool chung, có thể gặp throttling.')

In [ ]:
assert RUN_BENCHMARK, 'Đổi RUN_BENCHMARK=True để bắt đầu lượt chạy đã chọn'
if not RESUME and OUTPUT.exists() and any(OUTPUT.iterdir()):
    raise ValueError('Fresh run cần output mới/rỗng; chọn đường dẫn khác hoặc RESUME=True')
command = [sys.executable,'-m','arxiv_pipeline.reporting.milestone1.benchmark','--output',str(OUTPUT),'--results',str(RESULTS),'--',*args]
completed = subprocess.run(command,cwd=SOURCE)
print('Pipeline exit:', completed.returncode)
print('Đọc benchmark.json, statistics.json và logs; exit=0 không có nghĩa mọi version có source.')

## Kiểm tra lại và lưu bằng chứng

Sau khi pipeline dừng, revalidate trên snapshot đã hoàn tất. Chỉ dữ liệu summary được đưa vào book; không giả định lỗi ngắt đã có manifest mới. Nếu chưa có manifest, resume hoàn tất trước khi tổng hợp.

In [ ]:
report_path = RESULTS/'benchmark.json'
if report_path.exists():
    report = json.loads(report_path.read_text())
    if report.get('statistics_status') == 'current_run':
        # Never regenerate current-run statistics from a stale manifest after interruption.
        from arxiv_pipeline.reporting.milestone1.benchmark import current_snapshot
        current_snapshot(OUTPUT, report['run_id'])
        subprocess.run([sys.executable,'-m','arxiv_pipeline.reporting.milestone1.statistics',
            '--output',str(OUTPUT),'--save',str(RESULTS/'statistics-revalidated.json'),
            '--revalidate'],check=True,cwd=SOURCE)
        checked = json.loads((RESULTS/'statistics-revalidated.json').read_text())
        checked.update(label=report['label'], run_id=report['run_id'])
        (RESULTS/'statistics.json').write_text(json.dumps(checked,ensure_ascii=False,indent=2),encoding='utf-8')
        display(checked['rates'])
    else:
        print('No completed current-run snapshot; preserve raw evidence, do not publish old statistics.')
    report['environment']['is_colab'] = True
    report['environment']['cpu_only_confirmation'] = 'CPU-only selected; no NVIDIA GPU detected by notebook check'
    report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
# Audit every assigned lookup independently of the benchmark subset/snapshot.
audit_process = subprocess.run([sys.executable,'-m','arxiv_pipeline.reporting.milestone1.audit',
    '--output',str(OUTPUT),'--assignment',str(SOURCE/'configs/ms1_group6_ranges.json'),
    '--save',str(RESULTS/'assignment-audit.json')],cwd=SOURCE)
print('Assignment audit exit:', audit_process.returncode, '(1 means incomplete/error; inspect JSON)')
archive = shutil.make_archive(str(RESULTS), 'zip', RESULTS)
from google.colab import files
files.download(archive)